# IES Metrics Lab — Analysis Notebook

Load transcripts, run the evaluation pipeline, and explore how metrics behave across conditions and turns.

In [ ]:
import sys
import json
from pathlib import Path

# Add src to path when running from notebooks/
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from ies_lab import (
    MetricScorer,
    GroundTruthSearch,
    run_batch,
    run_batch_multiturn,
    load_all_transcripts,
)

sns.set_theme(style="whitegrid", palette="muted")

TRANSCRIPTS_DIR = REPO_ROOT / "transcripts"
RULES_PATH      = REPO_ROOT / "mapping_rules.yaml"
RUNS_DIR        = REPO_ROOT / "runs"
CACHE_PATH      = RUNS_DIR / "search_cache.json"

METRICS = ["FBS2", "EUS", "TCC", "NAI", "ABC", "CS", "SCS"]
FAILURE_TAGS = ["FB", "EV", "OC", "AM", "ID", "HC", "RJ", "NG", "DIF"]

# --- Configuration ---
USE_SEARCH = False   # Set True to enable web search grounding (requires network)
PRESET     = "default"  # Options: default, strict, lenient

print(f"Repo root: {REPO_ROOT}")
print(f"Transcripts dir: {TRANSCRIPTS_DIR}")

In [ ]:
# --- Run pipeline ---
search = GroundTruthSearch(cache_path=CACHE_PATH) if USE_SEARCH else None
scorer = MetricScorer(preset=PRESET, search=search)

batch_results     = run_batch(TRANSCRIPTS_DIR, RULES_PATH, scorer)
multiturn_results = run_batch_multiturn(TRANSCRIPTS_DIR, RULES_PATH, scorer)

print(f"Loaded {len(batch_results)} transcripts (final-turn mode)")
print(f"Loaded {len(multiturn_results)} turn evaluations (multi-turn mode)")

## 1. Summary Table

In [ ]:
rows = []
for r in batch_results:
    row = {
        "id":        r["transcript_id"],
        "condition": r.get("condition", ""),
        "failures":  ", ".join(r["failures"]) if r["failures"] else "—",
        "action":    r["action"] or "—",
    }
    row.update({m: r["metric_scores"].get(m, float("nan")) for m in METRICS})
    rows.append(row)

df = pd.DataFrame(rows)
pd.set_option("display.float_format", "{:.2f}".format)
df

## 2. Metric Score Distributions

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, metric in enumerate(METRICS):
    ax = axes[i]
    vals = df[metric].dropna()
    if len(vals) > 1:
        sns.histplot(vals, bins=10, ax=ax, kde=True)
    elif len(vals) == 1:
        ax.axvline(vals.iloc[0], color="steelblue", linewidth=2)
    ax.set_title(metric, fontweight="bold")
    ax.set_xlim(0, 1)
    ax.set_xlabel("Score")
    ax.set_ylabel("Count")

# Hide unused subplot
axes[-1].set_visible(False)
fig.suptitle("Metric Score Distributions (Final Turn)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Failure Rate Bar Chart

In [ ]:
tag_counts = {tag: 0 for tag in FAILURE_TAGS}
for r in batch_results:
    for tag in r["failures"]:
        if tag in tag_counts:
            tag_counts[tag] += 1

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(tag_counts.keys(), tag_counts.values(), color=sns.color_palette("muted", len(FAILURE_TAGS)))
ax.bar_label(bars)
ax.set_title("Failure Tag Frequency", fontweight="bold")
ax.set_xlabel("Failure Tag")
ax.set_ylabel("Count")
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

## 4. Metric Score Heatmap (Transcripts × Metrics)

In [ ]:
heat_df = df.set_index("id")[METRICS]

fig, ax = plt.subplots(figsize=(10, max(3, len(heat_df) * 0.6 + 1)))
sns.heatmap(
    heat_df.astype(float), annot=True, fmt=".2f",
    vmin=0, vmax=1, cmap="RdYlGn", linewidths=0.5, ax=ax
)
ax.set_title("Metric Scores — All Transcripts", fontweight="bold")
ax.set_xlabel("Metric")
ax.set_ylabel("Transcript ID")
plt.tight_layout()
plt.show()

## 5. Multi-Turn Metric Trajectories

In [ ]:
mt_rows = []
for r in multiturn_results:
    row = {
        "transcript_id": r["transcript_id"],
        "turn_index":    r["turn_index"],
        "condition":     r.get("condition", ""),
    }
    row.update({m: r["metric_scores"].get(m, float("nan")) for m in METRICS})
    mt_rows.append(row)

mt_df = pd.DataFrame(mt_rows)

fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey=True)
axes = axes.flatten()
palette = sns.color_palette("tab10")
transcript_ids = mt_df["transcript_id"].unique()

for i, metric in enumerate(METRICS):
    ax = axes[i]
    for j, tid in enumerate(transcript_ids):
        sub = mt_df[mt_df["transcript_id"] == tid].sort_values("turn_index")
        ax.plot(sub["turn_index"], sub[metric], marker="o", label=tid, color=palette[j % len(palette)])
    ax.set_title(metric, fontweight="bold")
    ax.set_xlabel("Turn")
    ax.set_ylabel("Score")
    ax.set_ylim(0, 1.05)
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.7, alpha=0.6)

axes[-1].set_visible(False)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower right", title="Transcript", ncol=1)
fig.suptitle("Metric Trajectories Across Turns", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Turn-Level Heatmap

In [ ]:
mt_df["label"] = mt_df["transcript_id"] + " T" + mt_df["turn_index"].astype(str)
turn_heat = mt_df.set_index("label")[METRICS]

fig, ax = plt.subplots(figsize=(10, max(3, len(turn_heat) * 0.55 + 1)))
sns.heatmap(
    turn_heat.astype(float), annot=True, fmt=".2f",
    vmin=0, vmax=1, cmap="RdYlGn", linewidths=0.5, ax=ax
)
ax.set_title("Turn-Level Metric Heatmap", fontweight="bold")
ax.set_xlabel("Metric")
ax.set_ylabel("Transcript / Turn")
plt.tight_layout()
plt.show()

## 7. Box Plots — Metric Scores by Failure Tag

In [ ]:
# Expand failures into boolean columns
for tag in FAILURE_TAGS:
    df[f"has_{tag}"] = df["failures"].str.contains(tag, regex=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, metric in enumerate(METRICS):
    ax = axes[i]
    # Build long-form data: for each tag that fired, what was the metric score?
    long_rows = []
    for _, row in df.iterrows():
        fired_tags = [t for t in FAILURE_TAGS if row.get(f"has_{t}", False)]
        if not fired_tags:
            long_rows.append({"tag": "none", "score": row[metric]})
        for tag in fired_tags:
            long_rows.append({"tag": tag, "score": row[metric]})
    long_df = pd.DataFrame(long_rows)
    if not long_df.empty:
        sns.boxplot(data=long_df, x="tag", y="score", ax=ax, palette="muted")
    ax.set_title(metric, fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("Failure Tag")
    ax.set_ylabel("Score")

axes[-1].set_visible(False)
fig.suptitle("Metric Scores by Failure Tag", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Condition Comparison

In [ ]:
if df["condition"].nunique() > 1:
    cond_means = df.groupby("condition")[METRICS].mean()
    fig, ax = plt.subplots(figsize=(10, max(3, len(cond_means) * 0.8 + 1)))
    sns.heatmap(
        cond_means, annot=True, fmt=".2f",
        vmin=0, vmax=1, cmap="RdYlGn", linewidths=0.5, ax=ax
    )
    ax.set_title("Mean Metric Scores by Condition", fontweight="bold")
    ax.set_xlabel("Metric")
    ax.set_ylabel("Condition")
    plt.tight_layout()
    plt.show()
else:
    print("Only one condition in dataset — add transcripts with different 'condition' values to enable this chart.")

## 9. Search Grounding Comparison

Only shown when `USE_SEARCH = True`.

In [ ]:
if USE_SEARCH:
    # Re-run without search for comparison
    scorer_nosearch = MetricScorer(preset=PRESET)
    batch_no_search = run_batch(TRANSCRIPTS_DIR, RULES_PATH, scorer_nosearch)

    rows_cmp = []
    for r_search, r_plain in zip(batch_results, batch_no_search):
        tid = r_search["transcript_id"]
        for m in METRICS:
            s_val = r_search["metric_scores"].get(m, float("nan"))
            p_val = r_plain["metric_scores"].get(m, float("nan"))
            rows_cmp.append({"id": tid, "metric": m, "with_search": s_val, "without_search": p_val, "delta": s_val - p_val})

    cmp_df = pd.DataFrame(rows_cmp)
    print("Score deltas (search-grounded minus heuristic-only):")
    pivot = cmp_df.pivot_table(index="id", columns="metric", values="delta")
    display(pivot.style.background_gradient(cmap="RdYlGn", vmin=-0.3, vmax=0.3).format("{:.2f}"))
else:
    print("Set USE_SEARCH = True in the setup cell to enable this comparison.")

## 10. Fine-Tuning Helper

Shows transcripts where the engine's actual failures differ from any `expected_failures` field.

In [ ]:
transcripts = load_all_transcripts(TRANSCRIPTS_DIR)
expected_map = {
    t["id"]: set(t.get("expected_failures", []))
    for t in transcripts
    if t.get("expected_failures")
}

mismatches = []
for r in batch_results:
    tid = r["transcript_id"]
    if tid not in expected_map:
        continue
    expected = expected_map[tid]
    actual   = set(r["failures"])
    if expected != actual:
        mismatches.append({
            "id":       tid,
            "expected": ", ".join(sorted(expected)) or "—",
            "actual":   ", ".join(sorted(actual))   or "—",
            "missed":   ", ".join(sorted(expected - actual)) or "—",
            "extra":    ", ".join(sorted(actual - expected)) or "—",
        })

if mismatches:
    print(f"{len(mismatches)} mismatch(es) found:")
    display(pd.DataFrame(mismatches))
else:
    print("No mismatches found (or no transcripts with expected_failures set).")

## 11. Research Portfolio — Evidence Base

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT / "tools"))

from audit_session import audit, EVIDENCE_DIR

# Load accumulated findings index
findings_index_path = EVIDENCE_DIR / "findings_index.json"

if findings_index_path.exists():
    with open(findings_index_path) as f:
        findings_index = json.load(f)
    fi_df = pd.DataFrame(findings_index)
    print(f"{len(fi_df)} sessions in evidence base")
    display(fi_df[["session_id", "system", "date", "condition", "failures", "action"]])
else:
    print("No evidence base yet. Run: python tools/audit_session.py <transcript>")

### 11b. Failure Frequency Across All Audited Sessions

In [ ]:
if findings_index_path.exists() and len(fi_df) > 0:
    from collections import Counter
    all_tags = [tag for row in fi_df["failures"] for tag in row]
    tag_counts = Counter(all_tags)

    if tag_counts:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Bar chart: failure frequency
        ax = axes[0]
        tags = list(tag_counts.keys())
        counts = list(tag_counts.values())
        bars = ax.bar(tags, counts, color=sns.color_palette("muted", len(tags)))
        ax.bar_label(bars)
        ax.set_title("Failure Tag Frequency — All Sessions", fontweight="bold")
        ax.set_xlabel("Failure Tag")
        ax.set_ylabel("Sessions")
        ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

        # Heatmap: failures by system
        ax2 = axes[1]
        if "system" in fi_df.columns and fi_df["system"].nunique() > 0:
            all_tag_labels = sorted(tag_counts.keys())
            systems = fi_df["system"].unique()
            heat_data = []
            for sys_name in systems:
                sys_rows = fi_df[fi_df["system"] == sys_name]
                sys_tags = [t for row in sys_rows["failures"] for t in row]
                sys_counts = Counter(sys_tags)
                heat_data.append([sys_counts.get(t, 0) for t in all_tag_labels])
            heat_df2 = pd.DataFrame(heat_data, index=systems, columns=all_tag_labels)
            sns.heatmap(heat_df2, annot=True, fmt="d", cmap="YlOrRd",
                        linewidths=0.5, ax=ax2)
            ax2.set_title("Failure Tags by System", fontweight="bold")
        else:
            ax2.set_visible(False)

        plt.tight_layout()
        plt.show()
    else:
        print("No failure tags recorded yet.")
else:
    print("No evidence base yet.")